# scClone2DR Tutorial

This notebook shows how to train scClone2DR on the publicly available AML data: https://doi.org/10.5281/zenodo.20035241.

## Prerequisites: Download the data

**Important**: Before running this tutorial, you must first generate synthetic test data by running the notebook:

Download the data from https://doi.org/10.5281/zenodo.20035241. Unzip the folder data_preprocessed_for_scclone2dr and place it in a data/ directory at the root of this repository

In [ ]:
import sys
import scclone2dr
import matplotlib.pyplot as plt
import numpy as np
from copy import deepcopy

In [ ]:
import os
# current working directory
os.getcwd()

## Initialize Real-Data Model
Set file paths and create the scClone2DR model instance for real data.

In [ ]:
path_rna = "../data/data_preprocessed_for_scclone2dr/metacells_hallmarks_phenograph/"
path_fastdrug = "../data/data_preprocessed_for_scclone2dr/pharmacoscopy.csv"
datamodule = scclone2dr.data.RealData(path_fastdrug=path_fastdrug, path_rna=path_rna)

In [ ]:
data_ref = datamodule.get_real_data(concentration_DMSO="200", concentration_drug="10")

## Train/Test Split and Training
Split the real dataset and train the model with L1/L2 regularization.

In [ ]:
idxs_train = [i for i in range(int(0.7*data_ref['N']))]
idxs_test = [i for i in range(data_ref['N']) if not(i in idxs_train)]

data_train, data_test, sample_names_train, sample_names_test = datamodule.get_real_data_split(idxs_train, idxs_test)

In [ ]:
from pathlib import Path
import numpy as np
from scclone2dr.pipeline import scClone2DRPipeline
from scclone2dr.trainer import Trainer, GuideType

# 1) Build the pipeline from the already-loaded real-data module.
trainer = Trainer(guide_type=GuideType.LOWRANK_MVN, rank=10)
pipeline = scClone2DRPipeline(
    data_source=datamodule,
    trainer=trainer,
    mode_nu="noise_correction",
    mode_theta="not shared decoupled",
)

# Ensure model topology is configured from the dataset metadata.
pipeline.model.configure(datamodule)

train_model = True
if train_model:
    # 2) Fit on the already prepared data_train dictionary.
    params_svi = pipeline.fit(
        data=data_train,
        penalty_l1=0.1,
        penalty_l2=0.1,
        lr=0.01,
        n_steps=2000,
    )

    # 3) Save learned parameters.
    ckpt_dir = Path("./checkpoints")
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    ckpt_path = ckpt_dir / "real_data_pipeline_run.npz"
    pipeline.save(ckpt_path)
else:
    ckpt_dir = Path("./checkpoints")
    ckpt_path = ckpt_dir / "real_data_pipeline_run.npz"

# 4) Load trained pipeline from disk.
pipeline_loaded = scClone2DRPipeline.from_file(
    ckpt_path,
    data_source=datamodule,
)
pipeline_loaded.model.configure(datamodule)

# 5) Sample from posterior (Monte Carlo).
posterior_results = pipeline_loaded.sample_posterior(
    data=data_test,
    idxs_sample_eval=idxs_test,
    nb_ites=100,
    dir_save=None,
    sample_names=sample_names_test,
    model_name="real_data_",
)

# 6) Optional posterior predictive sample from the generative model.
posterior_predictive_data, _ = pipeline_loaded.model.sampling(data_test, params=posterior_results['params'])

## Compute evaluation metrics

In [ ]:
evaluations = pipeline_loaded.evaluate(data_test, posterior_results['params'])

## Fold Change Scatter Plot
Compare predicted vs observed fold changes visually.

In [ ]:
plt.scatter(evaluations.fold_change_data, evaluations.fold_change_pred)

## Fraction Visualization
Show tumor fractions for the validation data.

In [ ]:
scclone2dr.plots.show_fractions(data_test, posterior_results['data'], idxdrug=0)

## Cell Count Visualization
Display predicted number of non-malignant cells in control wells.

In [ ]:
scclone2dr.plots.show_cells(data_test, posterior_results['data'])

## Proportion Visualization
Plot clone proportions inferred by the model.

In [ ]:
scclone2dr.plots.show_proportions(data_test, posterior_results['params'])

## Beta Effects
Inspect beta parameters to interpret drug effects.

In [ ]:
scclone2dr.plots.show_beta(data_test, posterior_results['params'])

## Count Scatter Plot
Visualize observed vs predicted counts.

In [ ]:
scclone2dr.plots.scatter_counts(data_test, posterior_results['data'])

## Survival Probabilities
Compute survival probabilities from single-cell features and plot them.

In [ ]:
scclone2dr.plots.survival_probabilities(data_test, posterior_results['params']['PI'].detach().numpy(), datamodule.cluster2clonelabel, idxdrug=0)